In [1]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def im_to_array(path): # note for 28×28 pixel images only
  # list of all files in a folder
  import os
  files = os.listdir(path)

  # read images
  all_img = []
  for file in files:
    img = cv.imread(path+'/'+file, cv.IMREAD_GRAYSCALE)
    all_img.append(img)
  all_img = np.array(all_img).reshape((-1, 28*28))
  return all_img

im_0 = im_to_array('tiny_mnist/0')
y_0 = np.zeros(30)

im_1 = im_to_array('tiny_mnist/1')
y_1 = np.ones(30)

im_2 = im_to_array('tiny_mnist/2')
y_2 = 2*np.ones(30)

X = np.concatenate((im_0, im_1, im_2))
y = np.concatenate((y_0, y_1, y_2))

print(X.shape)
print(y.shape)

(90, 784)
(90,)


In [3]:
# feature scaling
def scale(x):
  scal = (x-x.mean(axis=0))/(x.max(axis=0)-x.min(axis=0))
  return np.nan_to_num(scal)

x = scale(X)

C:\Users\ASUS TUF GAMING A15\AppData\Local\Temp\ipykernel_5912\2910391311.py:3: RuntimeWarning: invalid value encountered in divide
  scal = (x-x.mean(axis=0))/(x.max(axis=0)-x.min(axis=0))


In [4]:
import torch
import torch.optim as optim
import torch.nn as nn

tx = torch.tensor(x, dtype=torch.float32) # tensor(m, 784)
ty = torch.tensor(y, dtype=torch.long) # tensor(m, )
m = tx.size(0)

# model
def model(tx, th1, b1, th2, b2, th3, b3):
  # hid layer 1
  out = torch.matmul(tx, th1) + b1
  out = torch.sigmoid(out)

  # hid layer 2
  out = torch.matmul(out, th2) + b2
  out = torch.sigmoid(out)

  # out layer
  out = torch.matmul(out, th3) + b3
  return out

In [5]:
# Training

# initialize parameters
th1 = torch.rand(784, 25, requires_grad=True, dtype=torch.float32)
th2 = torch.rand(25, 15, requires_grad=True, dtype=torch.float32)
th3 = torch.rand(15, 3, requires_grad=True, dtype=torch.float32)
b1 = torch.rand(1, 25, requires_grad=True, dtype=torch.float32)
b2 = torch.rand(1, 15, requires_grad=True, dtype=torch.float32)
b3 = torch.rand(1, 3, requires_grad=True, dtype=torch.float32)
# optimizer
opt = optim.SGD([th1, th2, th3, b1, b2, b3], lr=1)

cost_func = nn.CrossEntropyLoss()

for iter in range(1, 10000):
  out = model(tx, th1, b1, th2, b2, th3, b3)

  # cost function
  cost = cost_func(out, ty)
  cost.backward()

  if iter%40==0:
    print(f'iter: {iter:02d} => cost: {cost.item():.4f}')

  # updat parameter
  opt.step()
  opt.zero_grad()

iter: 40 => cost: 1.8143
iter: 80 => cost: 1.8796
iter: 120 => cost: 1.8413
iter: 160 => cost: 1.7993
iter: 200 => cost: 1.7581
iter: 240 => cost: 1.6973
iter: 280 => cost: 1.5779
iter: 320 => cost: 1.5028
iter: 360 => cost: 1.4615
iter: 400 => cost: 1.4337
iter: 440 => cost: 1.4123
iter: 480 => cost: 1.3792
iter: 520 => cost: 1.3467
iter: 560 => cost: 1.2790
iter: 600 => cost: 1.2056
iter: 640 => cost: 1.1762
iter: 680 => cost: 1.1254
iter: 720 => cost: 1.1006
iter: 760 => cost: 1.0638
iter: 800 => cost: 1.0515
iter: 840 => cost: 1.0332
iter: 880 => cost: 1.0120
iter: 920 => cost: 0.9875
iter: 960 => cost: 0.9401
iter: 1000 => cost: 0.7596
iter: 1040 => cost: 0.5051
iter: 1080 => cost: 0.4549
iter: 1120 => cost: 0.4175
iter: 1160 => cost: 0.3850
iter: 1200 => cost: 0.3129
iter: 1240 => cost: 0.2407
iter: 1280 => cost: 0.2383
iter: 1320 => cost: 0.2131
iter: 1360 => cost: 0.2120
iter: 1400 => cost: 0.2114
iter: 1440 => cost: 0.2110
iter: 1480 => cost: 0.2107
iter: 1520 => cost: 0.2104


In [6]:
# Evaluate on training set
out = model(tx, th1, b1, th2, b2, th3, b3) #tensor(90, 3)
predict = torch.argmax(out, dim=1) #tesor(m)
accuracy = (predict==ty).sum()/m
print(accuracy.item())

0.9444444179534912


In [ ]:
# Prepare test set
im_0_test = im_to_array('tiny_mnist/test/0')
y_0_test = np.zeros(len(im_0_test))

im_1_test = im_to_array('tiny_mnist/test/1')
y_1_test = np.ones(len(im_1_test))

im_2_test = im_to_array('tiny_mnist/test/2')
y_2_test = 2*np.ones(len(im_2_test))

X_test = np.concatenate((im_0_test, im_1_test, im_2_test))
y_test = np.concatenate((y_0_test, y_1_test, y_2_test))

print(X_test.shape)
print(y_test.shape)

x_test = scale(X_test)



(15, 784)
(15,)


C:\Users\ASUS TUF GAMING A15\AppData\Local\Temp\ipykernel_5912\2910391311.py:3: RuntimeWarning: invalid value encountered in divide
  scal = (x-x.mean(axis=0))/(x.max(axis=0)-x.min(axis=0))


In [8]:
tx_test = torch.tensor(x_test, dtype=torch.float32) # tensor(15, 784)
ty_test = torch.tensor(y_test, dtype=torch.long) # tensor(15, )
m_test = tx_test.size(0)

In [9]:
# Evaluate on test set
out = model(tx_test, th1, b1, th2, b2, th3, b3) #tensor(90, 3)
predict = torch.argmax(out, dim=1) #tesor(m)
accuracy = (predict==ty_test).sum()/m_test
print(accuracy.item())

0.8666666746139526


In [10]:
# Second model
# Training

# initialize parameters
th1 = torch.rand(784, 64, requires_grad=True, dtype=torch.float32)
th2 = torch.rand(64, 32, requires_grad=True, dtype=torch.float32)
th3 = torch.rand(32, 3, requires_grad=True, dtype=torch.float32)
b1 = torch.rand(1, 64, requires_grad=True, dtype=torch.float32)
b2 = torch.rand(1, 32, requires_grad=True, dtype=torch.float32)
b3 = torch.rand(1, 3, requires_grad=True, dtype=torch.float32)
# optimizer
opt = optim.SGD([th1, th2, th3, b1, b2, b3], lr=0.1)

cost_func = nn.CrossEntropyLoss()

for iter in range(1, 10000):
  out = model(tx, th1, b1, th2, b2, th3, b3)

  # cost function
  cost = cost_func(out, ty)
  cost.backward()

  if iter%40==0:
    print(f'iter: {iter:02d} => cost: {cost.item():.4f}')

  # updat parameter
  opt.step()
  opt.zero_grad()

iter: 40 => cost: 1.0206
iter: 80 => cost: 0.9274
iter: 120 => cost: 0.8568
iter: 160 => cost: 0.8052
iter: 200 => cost: 0.7690
iter: 240 => cost: 0.7440
iter: 280 => cost: 0.7267
iter: 320 => cost: 0.7144
iter: 360 => cost: 0.7053
iter: 400 => cost: 0.6984
iter: 440 => cost: 0.6931
iter: 480 => cost: 0.6887
iter: 520 => cost: 0.6850
iter: 560 => cost: 0.6816
iter: 600 => cost: 0.6784
iter: 640 => cost: 0.6750
iter: 680 => cost: 0.6712
iter: 720 => cost: 0.6671
iter: 760 => cost: 0.6630
iter: 800 => cost: 0.6589
iter: 840 => cost: 0.6552
iter: 880 => cost: 0.6521
iter: 920 => cost: 0.6497
iter: 960 => cost: 0.6478
iter: 1000 => cost: 0.6461
iter: 1040 => cost: 0.6447
iter: 1080 => cost: 0.6434
iter: 1120 => cost: 0.6423
iter: 1160 => cost: 0.6412
iter: 1200 => cost: 0.6402
iter: 1240 => cost: 0.6393
iter: 1280 => cost: 0.6384
iter: 1320 => cost: 0.6375
iter: 1360 => cost: 0.6367
iter: 1400 => cost: 0.6359
iter: 1440 => cost: 0.6351
iter: 1480 => cost: 0.6343
iter: 1520 => cost: 0.6336


In [11]:
# Evaluate on training set
out = model(tx, th1, b1, th2, b2, th3, b3) #tensor(90, 3)
predict = torch.argmax(out, dim=1) #tesor(m)
accuracy = (predict==ty).sum()/m
print(accuracy.item())

0.7222222089767456


In [12]:
# Evaluate on test set
out = model(tx_test, th1, b1, th2, b2, th3, b3) #tensor(90, 3)
predict = torch.argmax(out, dim=1) #tesor(m)
accuracy = (predict==ty_test).sum()/m_test
print(accuracy.item())

0.6000000238418579


In [13]:
# Third model
# Training

# initialize parameters
th1 = torch.rand(784, 16, requires_grad=True, dtype=torch.float32)
th2 = torch.rand(16, 8, requires_grad=True, dtype=torch.float32)
th3 = torch.rand(8, 3, requires_grad=True, dtype=torch.float32)
b1 = torch.rand(1, 16, requires_grad=True, dtype=torch.float32)
b2 = torch.rand(1, 8, requires_grad=True, dtype=torch.float32)
b3 = torch.rand(1, 3, requires_grad=True, dtype=torch.float32)
# optimizer
opt = optim.SGD([th1, th2, th3, b1, b2, b3], lr=1)

cost_func = nn.CrossEntropyLoss()

for iter in range(1, 10000):
  out = model(tx, th1, b1, th2, b2, th3, b3)

  # cost function
  cost = cost_func(out, ty)
  cost.backward()

  if iter%40==0:
    print(f'iter: {iter:02d} => cost: {cost.item():.4f}')

  # updat parameter
  opt.step()
  opt.zero_grad()

iter: 40 => cost: 0.9834
iter: 80 => cost: 0.9288
iter: 120 => cost: 0.9023
iter: 160 => cost: 0.8706
iter: 200 => cost: 0.8326
iter: 240 => cost: 0.7889
iter: 280 => cost: 0.7448
iter: 320 => cost: 0.7185
iter: 360 => cost: 0.6819
iter: 400 => cost: 0.6613
iter: 440 => cost: 0.6049
iter: 480 => cost: 0.5849
iter: 520 => cost: 0.5794
iter: 560 => cost: 0.5636
iter: 600 => cost: 0.5341
iter: 640 => cost: 0.5120
iter: 680 => cost: 0.5040
iter: 720 => cost: 0.4880
iter: 760 => cost: 0.4842
iter: 800 => cost: 0.4835
iter: 840 => cost: 0.4821
iter: 880 => cost: 0.4805
iter: 920 => cost: 0.4798
iter: 960 => cost: 0.4686
iter: 1000 => cost: 0.4416
iter: 1040 => cost: 0.4415
iter: 1080 => cost: 0.4398
iter: 1120 => cost: 0.4388
iter: 1160 => cost: 0.4380
iter: 1200 => cost: 0.4374
iter: 1240 => cost: 0.4368
iter: 1280 => cost: 0.4363
iter: 1320 => cost: 0.4334
iter: 1360 => cost: 0.4009
iter: 1400 => cost: 0.3557
iter: 1440 => cost: 0.3485
iter: 1480 => cost: 0.3459
iter: 1520 => cost: 0.3234


In [14]:
# Evaluate on training set
out = model(tx, th1, b1, th2, b2, th3, b3) #tensor(90, 3)
predict = torch.argmax(out, dim=1) #tesor(m)
accuracy = (predict==ty).sum()/m
print(accuracy.item())

0.9444444179534912


In [15]:
# Evaluate on test set
out = model(tx_test, th1, b1, th2, b2, th3, b3) #tensor(90, 3)
predict = torch.argmax(out, dim=1) #tesor(m)
accuracy = (predict==ty_test).sum()/m_test
print(accuracy.item())

0.8666666746139526
